# TRELLIS + Auto-Rigging en Google Colab

<a href="https://colab.research.google.com/github/aqdriop/Trellis-models-rig/blob/v0.1.0/colab/TRELLIS_AutoRig_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

Convierte una imagen en un modelo 3D con [Microsoft TRELLIS](https://github.com/microsoft/TRELLIS) y, si lo activas, exporta un **GLB con esqueleto (armature) y pesos de skinning** generados con Blender en modo headless. La interfaz es la app Gradio de [`aqdriop/Trellis-models-rig`](https://github.com/aqdriop/Trellis-models-rig).

## Antes de empezar

1. **Elige una GPU**: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → Acelerador por hardware → GPU*. TRELLIS pide unos 16 GB de VRAM: una **T4** (15 GB) va justa; una **L4**, **A100** o **H100** va mejor. Las GPU Blackwell (runtime «G4») no son compatibles con este cuaderno.
2. Ejecuta las celdas **en orden** (*Entorno de ejecución → Ejecutar todas*, o ▶ una a una).
3. La **primera vez puede tardar 15–25 minutos**: se instala PyTorch 2.4 con CUDA 12.1, se compilan extensiones y se descarga Blender. Además, al arrancar la app por primera vez se descargan los pesos del modelo (varios GB).
4. Si la sesión de Colab se reinicia o caduca, hay que repetir los pasos 1–3.

> No se modifica el entorno de Colab: se crea uno aparte (Python 3.10 + PyTorch 2.4), porque el de Colab (Python 3.13, CUDA 13, numpy 2) no es compatible con TRELLIS.

## 1 · Descargar el código

Clona este repositorio (app Gradio, wrapper de rigging, script de Blender y scripts de instalación) en `/content`.

In [ ]:
#@title 1 · Descargar el código { display-mode: "form" }
#@markdown `REPO_REF` apunta a la versión de este cuaderno. Cámbialo a `main` para usar el código más reciente.
REPO_URL = "https://github.com/aqdriop/Trellis-models-rig.git"  #@param {type:"string"}
REPO_REF = "v0.1.0"  #@param {type:"string"}

import codecs
import os
import re
import shutil
import signal
import subprocess
import sys
import time
from pathlib import Path

WORK = Path("/content") if Path("/content").is_dir() else Path.cwd()
REPO_DIR = WORK / "Trellis-models-rig"
ENV_FILE = WORK / "trellis_colab_env.sh"


def run(cmd, cwd=None, env=None, on_output=None):
    """Ejecuta `cmd` mostrando su salida en vivo. Lanza RuntimeError si termina con error."""
    with subprocess.Popen(
        cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        bufsize=0, start_new_session=True,
    ) as proc:
        decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")

        def emit(text):
            if text:
                sys.stdout.write(text)
                sys.stdout.flush()
                if on_output:
                    on_output(text)

        try:
            while True:
                chunk = os.read(proc.stdout.fileno(), 4096)
                if not chunk:
                    break
                emit(decoder.decode(chunk))
            emit(decoder.decode(b"", final=True))
        except BaseException:
            # Botón ■ de Colab (KeyboardInterrupt) u otro error: detener todo el grupo de procesos.
            try:
                os.killpg(proc.pid, signal.SIGTERM)
                proc.wait(timeout=20)
            except Exception:
                try:
                    os.killpg(proc.pid, signal.SIGKILL)
                except Exception:
                    pass
            raise
        code = proc.wait()
    if code != 0:
        raise RuntimeError(f"El comando terminó con código {code}: {' '.join(map(str, cmd))}")


if shutil.which("nvidia-smi") is None or subprocess.run(["nvidia-smi", "-L"], capture_output=True).returncode != 0:
    raise RuntimeError(
        "No hay GPU. Ve a «Entorno de ejecución → Cambiar tipo de entorno de ejecución», "
        "elige una GPU (T4, L4 o A100) y vuelve a ejecutar esta celda."
    )

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)  # volver a ejecutar esta celda restablece el código
run(["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)])
run(["git", "-C", str(REPO_DIR), "-c", "advice.detachedHead=false", "checkout", "--quiet", REPO_REF])
run(["git", "-C", str(REPO_DIR), "log", "-1", "--format=Código listo: %h %s (%cd)", "--date=short"])

## 2 · Instalar TRELLIS, PyTorch y Blender

Ejecuta `colab/setup_colab.sh`, que instala en un entorno aislado:

- **Python 3.10** con [uv](https://docs.astral.sh/uv/) y **PyTorch 2.4.0 + CUDA 12.1** (+ xformers),
- un **toolkit CUDA 12.1** aparte (el `nvcc` de Colab es CUDA 13 y no sirve para compilar las extensiones de TRELLIS),
- `diff-gaussian-rasterization` (compilada para tu GPU), `nvdiffrast` y el resto de dependencias, **fijadas** en `colab/requirements-colab.lock`,
- el código de **TRELLIS** (commit fijado) y **Blender 4.5 LTS** (con verificación SHA-256).

Al terminar comprueba que todo importa y que la GPU responde. Si una fase falla, muestra las últimas líneas del registro (completo en `/content/setup_logs/`). Es seguro volver a ejecutarla: las fases ya hechas se omiten.

In [ ]:
#@title 2 · Instalar TRELLIS, PyTorch 2.4 y Blender (primera vez: 15–25 min) { display-mode: "form" }
started = time.time()
run(
    ["bash", str(REPO_DIR / "colab" / "setup_colab.sh")],
    cwd=str(REPO_DIR),
    env={**os.environ, "TRELLIS_COLAB_WORK": str(WORK)},
)
print(f"\nInstalación terminada en {(time.time() - started) / 60:.1f} min.")

## 3 · Iniciar la aplicación

Lanza `app.py` con el entorno de la celda anterior y un **enlace público de Gradio**. Cuando termine de cargar el modelo aparece un enlace `https://….gradio.live`: ábrelo en otra pestaña. **Esta celda queda en ejecución mientras la app esté activa**; pulsa ■ para detenerla.

In [ ]:
#@title 3 · Iniciar la aplicación { display-mode: "form" }
from IPython.display import HTML, display

_seen_urls = set()
_tail = ""


def mostrar_enlace(texto):
    """Destaca el enlace público de Gradio en cuanto aparece en la salida."""
    global _tail
    _tail = (_tail + texto)[-2000:]
    for url in re.findall(r"https://[A-Za-z0-9.-]+\.gradio\.live", _tail):
        if url not in _seen_urls:
            _seen_urls.add(url)
            display(HTML(
                '<div style="margin:8px 0;font:16px sans-serif">✅ Aplicación lista → '
                f'<a href="{url}" target="_blank" rel="noopener"><b>{url}</b></a></div>'
            ))


if not ENV_FILE.exists():
    raise RuntimeError("Falta el entorno: ejecuta primero el paso 2 (instalación).")
run(
    ["bash", "-c", f'source "{ENV_FILE}" && cd "{REPO_DIR}" && exec python app.py'],
    cwd=str(REPO_DIR),
    on_output=mostrar_enlace,
)

## Cómo usar la app

1. Abre el enlace `https://….gradio.live` que aparece al iniciar el paso 3.
2. Sube una imagen (mejor un objeto o personaje aislado, con fondo simple) y pulsa **Generate 3D**.
3. Ajusta *Simplify* y *Texture size*, marca **Enable Auto-Rigging** y pulsa **Extract GLB**.
4. Pulsa **Download GLB** y ábrelo en Blender, Godot, Unity… para revisar el esqueleto.

## Qué conviene saber

- **El rig es una plantilla humanoide aproximada**, ajustada a la caja envolvente de la malla; no detecta articulaciones. Es un punto de partida para personajes más o menos verticales: los objetos no humanoides pueden necesitar ajuste manual.
- **El enlace público lo puede usar cualquiera que lo tenga** mientras el paso 3 siga en ejecución (caduca a las 72 h). No lo compartas si no quieres que otros usen tu GPU.
- Colab es efímero: los modelos generados se pierden al cerrar la sesión. Descarga el GLB antes de parar.

## Problemas frecuentes

| Síntoma | Qué hacer |
| --- | --- |
| `No hay GPU` | *Entorno de ejecución → Cambiar tipo de entorno de ejecución* y elige una GPU. |
| `CUDA out of memory` | Baja *Texture size* a 512–1024, reduce los pasos de muestreo o usa una L4/A100. Si la memoria queda ocupada: *Entorno de ejecución → Reiniciar sesión* y repite los pasos 1–3. |
| La GPU es Blackwell («G4») | No compatible con PyTorch 2.4/CUDA 12.1. Cambia a T4, L4, A100 o H100. |
| Falla una fase de la instalación | Lee las últimas líneas que se muestran (registro completo en `/content/setup_logs/`) y vuelve a ejecutar el paso 2. Si Colab cambió algo de su entorno, abre un *issue* en el repositorio adjuntando ese registro. |
| El auto-rig dice que falta Blender | Vuelve a ejecutar el paso 2: instala Blender y define `BLENDER_BIN`. |